# Handling Dominant Features in XGBoost
### A Comprehensive Guide to Feature Importance, Feature Crossing, and Cardinality

---

## Table of Contents
1. **Feature Importance in XGBoost** — How it's computed and what it means
2. **Diagnosing Dominant Features** — Investigation framework
3. **Feature Crossing** — Concept, math, and when it's needed
4. **Feature Crossing for Tree-based Models** — Why trees learn interactions implicitly
5. **Techniques to Handle Dominant Features** — 8 practical approaches
6. **The High Cardinality Problem** — Why it causes false dominance
7. **Summary & Decision Framework**

---

## 1. Feature Importance in XGBoost

XGBoost computes feature importance using three metrics:

| Metric | Definition |
| --- | --- |
| **Gain** | Average reduction in the loss function when a feature is used for splitting |
| **Weight (Frequency)** | Number of times a feature appears in all trees |
| **Cover** | Average number of training samples affected by splits on this feature |

### Mathematical Foundation

XGBoost minimizes the regularized objective at iteration $$t$$:

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} l(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

where the regularization term is:

$$\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

Here $$T$$ is the number of leaves, $$w_j$$ is the weight of leaf $$j$$, $$\gamma$$ controls tree complexity, and $$\lambda$$ is the L2 regularization parameter.

The **gain** for a candidate split on feature $$k$$ at node $$m$$ is:

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

where $$G_L = \sum_{i \in I_L} g_i$$ (sum of first-order gradients in left child) and $$H_L = \sum_{i \in I_L} h_i$$ (sum of second-order gradients).

A feature's **total gain importance** is the sum of gain values across all splits that use it.

In [0]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Generate synthetic dataset with one deliberately dominant feature
n_samples = 5000

# Features
age = np.random.normal(35, 10, n_samples)
income = np.random.normal(50000, 15000, n_samples)  # This will be made dominant
education_years = np.random.randint(8, 22, n_samples)
credit_score = np.random.normal(650, 80, n_samples)
num_products = np.random.randint(1, 8, n_samples)
tenure_months = np.random.randint(1, 120, n_samples)

# Target: heavily dependent on income (making it dominant)
logit = (
    0.8 * ((income - 50000) / 15000) +   # Strong signal from income
    0.1 * ((age - 35) / 10) +              # Weak signal from age
    0.05 * ((credit_score - 650) / 80) +   # Very weak signal
    0.02 * ((education_years - 14) / 4) +  # Almost negligible
    0.01 * ((num_products - 4) / 2) +      # Almost negligible
    0.01 * ((tenure_months - 60) / 30)     # Almost negligible
)
prob = 1 / (1 + np.exp(-logit))
target = (np.random.rand(n_samples) < prob).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'education_years': education_years,
    'credit_score': credit_score,
    'num_products': num_products,
    'tenure_months': tenure_months,
    'churn': target
})

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['churn'].value_counts(normalize=True).to_string()}")
print(f"\nFeature Statistics:")
display(df.describe().round(2))

In [0]:
# Split data
features = ['age', 'income', 'education_years', 'credit_score', 'num_products', 'tenure_months']
X = df[features]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

# Plot all three importance types side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, imp_type in enumerate(['weight', 'gain', 'cover']):
    importance = model.get_booster().get_score(importance_type=imp_type)
    # Normalize
    total = sum(importance.values())
    importance = {k: v/total for k, v in importance.items()}
    
    sorted_imp = dict(sorted(importance.items(), key=lambda x: x[1]))
    axes[idx].barh(list(sorted_imp.keys()), list(sorted_imp.values()), color='steelblue')
    axes[idx].set_title(f'Feature Importance ({imp_type.upper()})', fontsize=12)
    axes[idx].set_xlabel('Normalized Importance')

plt.suptitle('XGBoost Feature Importance: Income Dominates All Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n\u26a0\ufe0f Notice: 'income' dominates across all importance metrics.")
print("This is BY DESIGN in our synthetic data - income has 80% of the signal.")
print("In real scenarios, you must investigate WHETHER such dominance is legitimate.")

## 2. Diagnosing Dominant Features — Investigation Framework

When one or two features dominate the feature importance plot, follow this **diagnostic checklist** before taking corrective action:

### Step 1: Check for Data Leakage

**Data leakage** occurs when the model has access to information that wouldn't be available at prediction time.

| Leakage Type | Example | Detection |
| --- | --- | --- |
| **Target leakage** | Using `is_churned_flag` to predict churn | Feature is a direct encoding of the target |
| **Future leakage** | Using `total_purchases_next_month` | Temporal ordering violation |
| **Proxy leakage** | Using `cancellation_reason` (only exists after cancellation) | Feature only populated for positive class |

### Step 2: Check for Proxy/ID Columns

High-cardinality identifiers (`user_id`, `transaction_id`, row indices) can appear dominant because they create perfect memorization splits.

### Step 3: Validate with Holdout Performance

Train two models:
- **Model A**: With the dominant feature
- **Model B**: Without the dominant feature

If Model B's performance is nearly identical, the feature's importance is inflated (likely due to cardinality or noise overfitting).

### Step 4: Cross-reference Importance Metrics

Compare **Gain** vs **SHAP** vs **Permutation Importance**. If they disagree significantly, the dominant feature may be a statistical artifact.

In [0]:
# Diagnostic: Compare model performance WITH and WITHOUT the dominant feature

# Model A: All features (including dominant 'income')
model_a = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
model_a.fit(X_train, y_train)
auc_a = roc_auc_score(y_test, model_a.predict_proba(X_test)[:, 1])

# Model B: Without 'income'
features_no_income = [f for f in features if f != 'income']
model_b = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
model_b.fit(X_train[features_no_income], y_train)
auc_b = roc_auc_score(y_test, model_b.predict_proba(X_test[features_no_income])[:, 1])

print("="*60)
print("DIAGNOSTIC: Performance With vs Without Dominant Feature")
print("="*60)
print(f"\nModel A (all features):        AUC-ROC = {auc_a:.4f}")
print(f"Model B (without 'income'):    AUC-ROC = {auc_b:.4f}")
print(f"Performance drop:              {((auc_a - auc_b)/auc_a)*100:.2f}%")
print(f"\n{'='*60}")

if (auc_a - auc_b) / auc_a > 0.05:
    print("\u2705 Significant performance drop → 'income' carries genuine signal.")
    print("   The dominance is REAL. Consider regularization or decomposition")
    print("   to reduce fragility, but don't discard the feature.")
else:
    print("\u26a0\ufe0f Minimal performance drop → 'income' importance may be inflated.")
    print("   Investigate cardinality bias or leakage.")

## 3. Feature Crossing

### Definition

**Feature crossing** is the technique of combining two or more features into a new synthetic feature by computing their interaction (cartesian product for categoricals, or multiplication for numericals). It allows models to learn non-linear relationships that individual features alone cannot capture.

### Mathematical Formulation

For a linear model with features $$x_1$$ and $$x_2$$, the prediction is:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2$$

This can only model a **hyperplane** — it cannot capture the interaction between $$x_1$$ and $$x_2$$.

With a feature cross $$x_{1 \times 2} = x_1 \cdot x_2$$, the model becomes:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + w_3 (x_1 \cdot x_2)$$

Now it can model interaction effects.

### Types of Feature Crosses

| Cross Type | Formula | Example |
| --- | --- | --- |
| **Numerical × Numerical** | $$x_{cross} = x_1 \cdot x_2$$ | `price × quantity = revenue` |
| **Categorical × Categorical** | $$x_{cross} = \text{concat}(x_1, x_2)$$ | `country_language = "IN_en"` |
| **Numerical × Categorical** | One-hot encode categorical, multiply with numerical | `income × is_urban` |
| **Binned Numerical × Binned Numerical** | Bin both, then cross categories | `age_bucket × income_bucket` |

### When Feature Crossing Helps Most

1. **Linear models** (Logistic Regression, Linear SVM) — cannot learn interactions without explicit crosses
2. **Recommendation systems** — user × item crosses capture co-occurrence
3. **Geo-temporal patterns** — `hour × day_of_week`, `latitude × longitude` bucket
4. **Sparse categorical data** — where specific combinations carry unique signals

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures

# Demonstrate feature crossing with a non-linear decision boundary
# XOR-like problem: target is 1 when (x1 > 0 AND x2 > 0) OR (x1 < 0 AND x2 < 0)
np.random.seed(42)
n = 1000

x1 = np.random.randn(n)
x2 = np.random.randn(n)
y_xor = ((x1 > 0) & (x2 > 0)) | ((x1 < 0) & (x2 < 0))
y_xor = y_xor.astype(int)

df_xor = pd.DataFrame({'x1': x1, 'x2': x2, 'target': y_xor})

# Model 1: Logistic Regression WITHOUT feature cross
lr_no_cross = LogisticRegression(random_state=42)
lr_no_cross.fit(df_xor[['x1', 'x2']], y_xor)
auc_no_cross = roc_auc_score(y_xor, lr_no_cross.predict_proba(df_xor[['x1', 'x2']])[:, 1])

# Model 2: Logistic Regression WITH feature cross (x1 * x2)
df_xor['x1_x_x2'] = df_xor['x1'] * df_xor['x2']  # The magic cross!
lr_with_cross = LogisticRegression(random_state=42)
lr_with_cross.fit(df_xor[['x1', 'x2', 'x1_x_x2']], y_xor)
auc_with_cross = roc_auc_score(y_xor, lr_with_cross.predict_proba(df_xor[['x1', 'x2', 'x1_x_x2']])[:, 1])

# Model 3: XGBoost WITHOUT explicit cross (learns it implicitly)
xgb_no_cross = xgb.XGBClassifier(n_estimators=50, max_depth=3, random_state=42, eval_metric='logloss')
xgb_no_cross.fit(df_xor[['x1', 'x2']], y_xor)
auc_xgb = roc_auc_score(y_xor, xgb_no_cross.predict_proba(df_xor[['x1', 'x2']])[:, 1])

print("="*65)
print("FEATURE CROSSING DEMONSTRATION: XOR-like Problem")
print("="*65)
print(f"\nThe target is 1 when x1 and x2 have the SAME sign (interaction).")
print(f"A linear model cannot learn this without the cross term x1*x2.\n")
print(f"{'Model':<45} {'AUC-ROC':>8}")
print(f"{'-'*55}")
print(f"{'Logistic Regression (no cross)':<45} {auc_no_cross:>8.4f}")
print(f"{'Logistic Regression (with x1×x2 cross)':<45} {auc_with_cross:>8.4f}")
print(f"{'XGBoost (no explicit cross, depth=3)':<45} {auc_xgb:>8.4f}")
print(f"\n{'='*65}")
print(f"\n→ Logistic Regression NEEDS the cross to solve this problem.")
print(f"→ XGBoost learns the interaction implicitly via sequential splits.")

In [0]:
# Visualize the decision boundaries
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Create mesh grid
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

# Plot 1: LogReg without cross
Z1 = lr_no_cross.predict_proba(grid)[:, 1].reshape(xx.shape)
axes[0].contourf(xx, yy, Z1, levels=20, cmap='RdBu', alpha=0.7)
axes[0].scatter(x1[y_xor==1], x2[y_xor==1], c='blue', s=5, alpha=0.3, label='Class 1')
axes[0].scatter(x1[y_xor==0], x2[y_xor==0], c='red', s=5, alpha=0.3, label='Class 0')
axes[0].set_title(f'LogReg (No Cross)\nAUC={auc_no_cross:.3f}', fontsize=11)
axes[0].set_xlabel('x1'); axes[0].set_ylabel('x2')
axes[0].legend()

# Plot 2: LogReg with cross
grid_cross = np.c_[grid, grid[:, 0] * grid[:, 1]]
Z2 = lr_with_cross.predict_proba(grid_cross)[:, 1].reshape(xx.shape)
axes[1].contourf(xx, yy, Z2, levels=20, cmap='RdBu', alpha=0.7)
axes[1].scatter(x1[y_xor==1], x2[y_xor==1], c='blue', s=5, alpha=0.3, label='Class 1')
axes[1].scatter(x1[y_xor==0], x2[y_xor==0], c='red', s=5, alpha=0.3, label='Class 0')
axes[1].set_title(f'LogReg (With x1×x2 Cross)\nAUC={auc_with_cross:.3f}', fontsize=11)
axes[1].set_xlabel('x1'); axes[1].set_ylabel('x2')
axes[1].legend()

# Plot 3: XGBoost
Z3 = xgb_no_cross.predict_proba(grid)[:, 1].reshape(xx.shape)
axes[2].contourf(xx, yy, Z3, levels=20, cmap='RdBu', alpha=0.7)
axes[2].scatter(x1[y_xor==1], x2[y_xor==1], c='blue', s=5, alpha=0.3, label='Class 1')
axes[2].scatter(x1[y_xor==0], x2[y_xor==0], c='red', s=5, alpha=0.3, label='Class 0')
axes[2].set_title(f'XGBoost (No Explicit Cross)\nAUC={auc_xgb:.3f}', fontsize=11)
axes[2].set_xlabel('x1'); axes[2].set_ylabel('x2')
axes[2].legend()

plt.suptitle('Feature Crossing: Linear Model vs Tree-Based Model on XOR Problem', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n\u27a1\ufe0f Key Insight: Trees capture the quadrant structure via splits on x1 then x2.")
print("   Linear models see only a flat hyperplane without the explicit cross term.")

## 4. Feature Crossing for Tree-Based Models — Why It's Usually Unnecessary

### How Trees Learn Interactions Implicitly

A decision tree captures an interaction between features $$A$$ and $$B$$ through **sequential splits**:

```
        [Root]
       /      \
    A < 5     A >= 5
    /   \      /    \
  B<3  B>=3  B<3  B>=3
  ↓     ↓     ↓     ↓
 leaf1 leaf2 leaf3 leaf4
```

This structure effectively partitions the feature space into 4 regions based on the $$A \times B$$ interaction — **without** an explicit cross feature.

### When Explicit Crosses Might Still Help Trees

| Scenario | Why it helps |
| --- | --- |
| **Shallow trees** (low `max_depth`) | Not enough depth to reach deep interactions |
| **Extremely sparse interactions** | Rare combinations need a direct signal |
| **Training speed** | Pre-computed cross reduces split search time |
| **Boosting with low learning rate** | Each tree is weak; crosses give a head start |

### Mathematical Intuition

For a depth-$$d$$ tree, the maximum number of feature interactions it can model is:

$$\text{Max interactions} = \binom{p}{d}$$

where $$p$$ is the number of features. With $$d=2$$, only pairwise interactions are captured per tree. In an **ensemble** of $$T$$ trees, the total capacity grows, but individual trees remain limited.

### Recommendation

> **For XGBoost/LightGBM with sufficient depth (≥5):** Explicit feature crosses are rarely needed.  
> **For shallow ensembles or linear models:** Feature crosses provide substantial lift.

In [0]:
# Prove that XGBoost performance doesn't improve with explicit crosses
# when given sufficient depth

from sklearn.preprocessing import PolynomialFeatures

# Use our original churn dataset
X_train_base = X_train.copy()
X_test_base = X_test.copy()

# Create polynomial features (all pairwise interactions)
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = pd.DataFrame(
    poly.fit_transform(X_train_base),
    columns=poly.get_feature_names_out(features)
)
X_test_poly = pd.DataFrame(
    poly.transform(X_test_base),
    columns=poly.get_feature_names_out(features)
)

results = []

# Test across different tree depths
for depth in [2, 3, 5, 8]:
    # Without crosses
    m1 = xgb.XGBClassifier(n_estimators=100, max_depth=depth, learning_rate=0.1, 
                           random_state=42, eval_metric='logloss')
    m1.fit(X_train_base, y_train)
    auc1 = roc_auc_score(y_test, m1.predict_proba(X_test_base)[:, 1])
    
    # With all pairwise crosses
    m2 = xgb.XGBClassifier(n_estimators=100, max_depth=depth, learning_rate=0.1, 
                           random_state=42, eval_metric='logloss')
    m2.fit(X_train_poly, y_train)
    auc2 = roc_auc_score(y_test, m2.predict_proba(X_test_poly)[:, 1])
    
    results.append({'max_depth': depth, 'AUC (no cross)': auc1, 
                    'AUC (with crosses)': auc2, 'Improvement': auc2 - auc1})

results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("EXPERIMENT: Do Explicit Feature Crosses Help XGBoost?")
print("="*70)
print(f"\nOriginal features: {len(features)} | With pairwise crosses: {X_train_poly.shape[1]}")
print(f"\n{'max_depth':<12} {'AUC (no cross)':<18} {'AUC (with crosses)':<20} {'\u0394 AUC':>8}")
print("-"*60)
for _, row in results_df.iterrows():
    print(f"{int(row['max_depth']):<12} {row['AUC (no cross)']:<18.4f} {row['AUC (with crosses)']:<20.4f} {row['Improvement']:>+8.4f}")

print(f"\n{'='*70}")
print("\n\u2705 Conclusion: For depth ≥ 5, explicit crosses provide negligible improvement.")
print("   Trees already discover these interactions through sequential splits.")
print("   At depth=2, crosses help slightly because the tree is too shallow.")

## 5. Techniques to Handle Dominant Features

When a feature legitimately dominates but you want a **more robust, diversified model**, apply one or more of these techniques:

| # | Technique | Reduces Importance By | Best When |
| --- | --- | --- | --- |
| 5.1 | Feature Decomposition | Breaking one feature into sub-signals | Feature is an aggregate of distinct concepts |
| 5.2 | Regularization Tuning | Penalizing over-reliance | You want the feature but constrained |
| 5.3 | Feature Binning | Capping information content | Continuous dominant feature |
| 5.4 | Target Encoding with Smoothing | Compressing cardinality | High-cardinality categorical dominant |
| 5.5 | Residual Modeling (Two-Stage) | Isolating the dominant signal | You want to understand "what else matters" |
| 5.6 | Feature Importance Constraints | Hard limits on usage | Production robustness requirement |
| 5.7 | Dimensionality Reduction (PCA) | Distributing variance evenly | Correlated feature group |
| 5.8 | SHAP-based Analysis | Better measurement (not correction) | Suspecting measurement bias |

---

### 5.1 Feature Decomposition

Instead of one monolithic feature, break it into sub-components that distribute the signal:

$$\text{income} \longrightarrow \begin{cases} \text{income\_base\_salary} \\ \text{income\_bonus} \\ \text{income\_percentile\_in\_zip} \\ \text{income\_growth\_rate} \end{cases}$$

Each sub-feature carries a portion of the original signal, preventing any single one from dominating.

In [0]:
# 5.1 Feature Decomposition: Break 'income' into sub-components

df_decomposed = df.copy()

# Decompose income into multiple informative sub-features
df_decomposed['income_percentile'] = df_decomposed['income'].rank(pct=True)
df_decomposed['income_z_score'] = (df_decomposed['income'] - df_decomposed['income'].mean()) / df_decomposed['income'].std()
df_decomposed['income_bracket'] = pd.qcut(df_decomposed['income'], q=5, labels=[1,2,3,4,5]).astype(int)
df_decomposed['income_to_age_ratio'] = df_decomposed['income'] / (df_decomposed['age'] + 1)
df_decomposed['income_above_median'] = (df_decomposed['income'] > df_decomposed['income'].median()).astype(int)

# Train with decomposed features (remove original 'income')
features_decomposed = ['age', 'income_percentile', 'income_z_score', 'income_bracket', 
                       'income_to_age_ratio', 'income_above_median',
                       'education_years', 'credit_score', 'num_products', 'tenure_months']

X_dec = df_decomposed[features_decomposed]
y_dec = df_decomposed['churn']
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_dec, y_dec, test_size=0.2, random_state=42)

model_decomposed = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, 
                                      random_state=42, eval_metric='logloss')
model_decomposed.fit(X_train_d, y_train_d)
auc_decomposed = roc_auc_score(y_test_d, model_decomposed.predict_proba(X_test_d)[:, 1])

# Compare importance distribution
importance_dec = model_decomposed.get_booster().get_score(importance_type='gain')
total_imp = sum(importance_dec.values())
importance_dec = {k: v/total_imp for k, v in importance_dec.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original
imp_orig = model.get_booster().get_score(importance_type='gain')
total_orig = sum(imp_orig.values())
imp_orig = {k: v/total_orig for k, v in imp_orig.items()}
sorted_orig = dict(sorted(imp_orig.items(), key=lambda x: x[1]))
axes[0].barh(list(sorted_orig.keys()), list(sorted_orig.values()), color='coral')
axes[0].set_title('BEFORE: Original Features\n(income dominates)', fontsize=11)
axes[0].set_xlabel('Normalized Gain')
axes[0].axvline(x=1/len(imp_orig), color='gray', linestyle='--', alpha=0.5, label='Equal share')
axes[0].legend()

# Decomposed
sorted_dec = dict(sorted(importance_dec.items(), key=lambda x: x[1]))
axes[1].barh(list(sorted_dec.keys()), list(sorted_dec.values()), color='seagreen')
axes[1].set_title('AFTER: Decomposed Features\n(signal distributed)', fontsize=11)
axes[1].set_xlabel('Normalized Gain')
axes[1].axvline(x=1/len(importance_dec), color='gray', linestyle='--', alpha=0.5, label='Equal share')
axes[1].legend()

plt.suptitle('Feature Decomposition: Distributing the Dominant Signal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

max_orig = max(imp_orig.values())
max_dec = max(importance_dec.values())
print(f"\nMax feature importance (original):    {max_orig:.3f} (income)")
print(f"Max feature importance (decomposed):  {max_dec:.3f}")
print(f"AUC (original): {auc_a:.4f} | AUC (decomposed): {auc_decomposed:.4f}")
print(f"\n✅ Importance is now distributed across income sub-features.")
print(f"   No single feature exceeds {max_dec:.1%} of total importance.")

### 5.2 Regularization Tuning

XGBoost's regularization parameters directly control how much any single feature can dominate:

| Parameter | Effect on Dominance | Mathematical Role |
| --- | --- | --- |
| `reg_lambda` (L2) | Shrinks leaf weights, reducing extreme splits | $$\Omega(f) = \frac{1}{2}\lambda \sum w_j^2$$ |
| `reg_alpha` (L1) | Encourages sparsity in leaf weights | $$\Omega(f) = \alpha \sum |w_j|$$ |
| `colsample_bytree` | Forces some trees to NOT use the dominant feature | Probability of feature exclusion |
| `colsample_bylevel` | At each depth level, randomly excludes features | Finer-grained feature dropout |
| `max_depth` | Limits how deeply the model can exploit one feature | Interaction capacity cap |
| `gamma` (min split gain) | Requires higher gain for each split, penalizing marginal ones | $$\text{Gain} - \gamma > 0$$ |

The key insight: **`colsample_bytree = 0.5`** means that in 50% of trees, the dominant feature is unavailable, forcing the model to learn from other features.

In [0]:
# 5.2 Regularization Tuning: Show how parameters redistribute importance

configs = {
    'Baseline (no reg)': {'reg_alpha': 0, 'reg_lambda': 1, 'colsample_bytree': 1.0},
    'L2 Heavy (lambda=10)': {'reg_alpha': 0, 'reg_lambda': 10, 'colsample_bytree': 1.0},
    'L1 Heavy (alpha=5)': {'reg_alpha': 5, 'reg_lambda': 1, 'colsample_bytree': 1.0},
    'Feature Dropout (col=0.5)': {'reg_alpha': 0, 'reg_lambda': 1, 'colsample_bytree': 0.5},
    'Combined Regularization': {'reg_alpha': 3, 'reg_lambda': 5, 'colsample_bytree': 0.6},
}

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5), sharey=True)

print("="*70)
print("REGULARIZATION EXPERIMENT: Impact on Feature Importance Distribution")
print("="*70)
print(f"\n{'Config':<30} {'AUC':>6} {'Max Importance':>15} {'Income %':>10}")
print("-"*65)

for idx, (name, params) in enumerate(configs.items()):
    m = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                          random_state=42, eval_metric='logloss', **params)
    m.fit(X_train, y_train)
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    
    imp = m.get_booster().get_score(importance_type='gain')
    total = sum(imp.values())
    imp_norm = {k: v/total for k, v in imp.items()}
    
    income_pct = imp_norm.get('income', 0)
    max_imp = max(imp_norm.values())
    
    sorted_imp = dict(sorted(imp_norm.items(), key=lambda x: x[1]))
    colors = ['coral' if k == 'income' else 'steelblue' for k in sorted_imp.keys()]
    axes[idx].barh(list(sorted_imp.keys()), list(sorted_imp.values()), color=colors)
    axes[idx].set_title(f'{name}\nAUC={auc:.3f}', fontsize=9)
    axes[idx].set_xlim(0, 1)
    
    print(f"{name:<30} {auc:>6.4f} {max_imp:>15.3f} {income_pct:>10.1%}")

plt.suptitle('Regularization Impact on Feature Importance (red = income)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n\u2705 'colsample_bytree' is the most effective at reducing single-feature dominance.")
print(f"   It forces the ensemble to learn patterns WITHOUT the dominant feature in ~50% of trees.")

### 5.3 Feature Binning / Discretization

Binning converts a continuous feature into a set of discrete categories, **capping its information content**.

#### How it reduces dominance:

A continuous feature with range $$[0, 100000]$$ offers thousands of potential split points. After binning into $$k$$ quantiles, it offers only $$k-1$$ splits — dramatically limiting the tree's ability to extract fine-grained signal from it.

$$x_{\text{binned}} = \text{quantile\_bin}(x, k) \in \{1, 2, ..., k\}$$

#### Types of Binning:

| Method | Formula | Properties |
| --- | --- | --- |
| **Equal-width** | $$\text{bin}_i = \lfloor k \cdot \frac{x - x_{min}}{x_{max} - x_{min}} \rfloor$$ | Sensitive to outliers |
| **Quantile (equal-frequency)** | Each bin has $$n/k$$ samples | Robust to skew |
| **Custom thresholds** | Domain-driven (e.g., income brackets) | Most interpretable |

In [0]:
# 5.3 Feature Binning: Cap the information content of the dominant feature
from sklearn.preprocessing import KBinsDiscretizer

df_binned = df.copy()

# Apply different binning strategies to 'income'
for n_bins in [3, 5, 10, 20]:
    binner = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
    df_binned[f'income_{n_bins}bins'] = binner.fit_transform(df_binned[['income']]).astype(int)

# Compare: continuous income vs binned income
results_bin = []

# Continuous (baseline)
m_cont = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
m_cont.fit(X_train, y_train)
auc_cont = roc_auc_score(y_test, m_cont.predict_proba(X_test)[:, 1])
imp_cont = m_cont.get_booster().get_score(importance_type='gain')
t = sum(imp_cont.values())
results_bin.append({'Variant': 'Continuous income', 'AUC': auc_cont, 
                    'Income Importance %': imp_cont.get('income', 0)/t})

# Binned variants
for n_bins in [3, 5, 10, 20]:
    features_bin = ['age', f'income_{n_bins}bins', 'education_years', 'credit_score', 'num_products', 'tenure_months']
    X_b = df_binned[features_bin]
    X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_b, df_binned['churn'], test_size=0.2, random_state=42)
    
    m_bin = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
    m_bin.fit(X_tr_b, y_tr_b)
    auc_bin = roc_auc_score(y_te_b, m_bin.predict_proba(X_te_b)[:, 1])
    imp_bin = m_bin.get_booster().get_score(importance_type='gain')
    t_bin = sum(imp_bin.values())
    income_key = f'income_{n_bins}bins'
    results_bin.append({'Variant': f'Income ({n_bins} bins)', 'AUC': auc_bin,
                        'Income Importance %': imp_bin.get(income_key, 0)/t_bin})

results_bin_df = pd.DataFrame(results_bin)
print("\n" + "="*60)
print("BINNING EXPERIMENT: Impact on Dominance vs Performance")
print("="*60)
print(f"\n{'Variant':<25} {'AUC':>8} {'Income Importance':>18}")
print("-"*55)
for _, row in results_bin_df.iterrows():
    print(f"{row['Variant']:<25} {row['AUC']:>8.4f} {row['Income Importance %']:>17.1%}")

print(f"\n\u2705 Fewer bins = less dominance, but also less predictive power.")
print(f"   The 5-10 bin range is typically the sweet spot: reduces dominance")
print(f"   while retaining most of the signal.")

### 5.4 Target Encoding with Smoothing

For high-cardinality categorical features, **target encoding** replaces each category with a smoothed estimate of the target mean for that category.

#### Formula

$$\text{TE}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{\text{global}}}{n_c + m}$$

where:
- $$n_c$$ = number of samples in category $$c$$
- $$\bar{y}_c$$ = mean target value for category $$c$$
- $$\bar{y}_{\text{global}}$$ = global target mean
- $$m$$ = smoothing factor (higher $$m$$ = more regularization toward global mean)

#### Why it reduces dominance:

| Before (one-hot) | After (target encoding) |
| --- | --- |
| 10,000 sparse binary columns | 1 continuous column |
| Each category = potential split point | Compressed signal, fewer splits |
| Memorization risk | Smoothing prevents overfitting |

#### Key Insight on Smoothing:

When $$n_c$$ is small (rare category), the encoding is pulled toward $$\bar{y}_{\text{global}}$$ (shrinkage). When $$n_c$$ is large, the category-specific mean dominates. This is **Bayesian shrinkage** in disguise.

In [0]:
# 5.4 Target Encoding with Smoothing

# Create a high-cardinality feature to demonstrate
np.random.seed(42)
n_categories = 200
df_te = df.copy()
df_te['merchant_id'] = np.random.choice([f'merchant_{i:04d}' for i in range(n_categories)], size=len(df_te))

# Manual target encoding with smoothing
def target_encode_smooth(df, col, target, smoothing=10):
    """Apply target encoding with Bayesian smoothing."""
    global_mean = df[target].mean()
    agg = df.groupby(col)[target].agg(['mean', 'count'])
    # Smoothed encoding: (count * category_mean + m * global_mean) / (count + m)
    smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    return df[col].map(smooth)

# Apply target encoding with different smoothing levels
for m in [1, 10, 50, 200]:
    df_te[f'merchant_te_m{m}'] = target_encode_smooth(df_te, 'merchant_id', 'churn', smoothing=m)

# Visualize the effect of smoothing
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
global_mean = df_te['churn'].mean()

for idx, m in enumerate([1, 10, 50, 200]):
    col = f'merchant_te_m{m}'
    axes[idx].hist(df_te[col], bins=30, color='teal', alpha=0.7, edgecolor='black')
    axes[idx].axvline(global_mean, color='red', linestyle='--', label=f'Global mean={global_mean:.2f}')
    axes[idx].set_title(f'Smoothing m={m}\nStd={df_te[col].std():.4f}', fontsize=10)
    axes[idx].set_xlabel('Encoded Value')
    axes[idx].legend(fontsize=8)

plt.suptitle('Target Encoding: Higher Smoothing → Less Variance → Less Dominance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n\u2705 With m=1 (low smoothing): High variance → feature dominates (overfits to rare categories)")
print(f"   With m=200 (high smoothing): All values collapse toward {global_mean:.2f} → feature becomes uninformative")
print(f"   Sweet spot (m=10-50): Balances signal retention with dominance reduction.")

### 5.5 Residual Modeling (Two-Stage Approach)

This technique explicitly separates "what the dominant feature explains" from "what else matters":

#### Algorithm:

1. **Stage 1**: Train a simple model using ONLY the dominant feature:
$$\hat{y}_{\text{stage1}} = f_1(x_{\text{dominant}})$$

2. **Compute residuals**: The unexplained signal:
$$r_i = y_i - \hat{y}_{\text{stage1}}(x_i)$$

3. **Stage 2**: Train a second model on ALL OTHER features, predicting the residuals:
$$\hat{r} = f_2(x_{\text{other features}})$$

4. **Final prediction**: Combine both stages:
$$\hat{y}_{\text{final}} = \hat{y}_{\text{stage1}} + \hat{y}_{\text{stage2}}$$

#### Why this works:

Stage 2 is forced to learn patterns that the dominant feature **cannot** explain. This reveals the unique contribution of each remaining feature, independent of the dominant one.

#### Analogy: Regression in statistics

This is equivalent to "partialing out" a confounding variable — examining the relationship between other features and the target **after controlling for** the dominant feature.

In [0]:
# 5.5 Residual Modeling: Two-Stage approach
from sklearn.calibration import CalibratedClassifierCV

# Stage 1: Model using ONLY the dominant feature
model_stage1 = xgb.XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, 
                                  random_state=42, eval_metric='logloss')
model_stage1.fit(X_train[['income']], y_train)

# Get Stage 1 predictions (probabilities)
stage1_probs_train = model_stage1.predict_proba(X_train[['income']])[:, 1]
stage1_probs_test = model_stage1.predict_proba(X_test[['income']])[:, 1]

auc_stage1 = roc_auc_score(y_test, stage1_probs_test)

# Compute residuals (what income alone can't explain)
residuals_train = y_train.values - stage1_probs_train

# Stage 2: Model using ALL OTHER features, predicting residuals
features_stage2 = [f for f in features if f != 'income']

model_stage2 = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1,
                                 random_state=42, objective='reg:squarederror')
model_stage2.fit(X_train[features_stage2], residuals_train)

# Stage 2 predictions
stage2_preds_test = model_stage2.predict(X_test[features_stage2])

# Combined prediction
final_preds = stage1_probs_test + stage2_preds_test
final_preds = np.clip(final_preds, 0, 1)  # Clip to valid probability range
auc_combined = roc_auc_score(y_test, final_preds)

# Feature importance of Stage 2 (what matters BEYOND income)
imp_stage2 = model_stage2.get_booster().get_score(importance_type='gain')
total_s2 = sum(imp_stage2.values())
imp_stage2_norm = {k: v/total_s2 for k, v in imp_stage2.items()}

print("="*65)
print("RESIDUAL MODELING: Two-Stage Approach")
print("="*65)
print(f"\nStage 1 (income only):          AUC = {auc_stage1:.4f}")
print(f"Full model (all features):      AUC = {auc_a:.4f}")
print(f"Two-stage combined:             AUC = {auc_combined:.4f}")
print(f"\n{'='*65}")
print(f"\nStage 2 Feature Importance (what matters BEYOND income):")
print(f"{'-'*45}")
for feat, imp in sorted(imp_stage2_norm.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 40)
    print(f"  {feat:<20} {imp:.3f} {bar}")

print(f"\n\u2705 Stage 2 reveals the 'hidden' importance structure.")
print(f"   These are features that add value BEYOND what income provides.")

### 5.6 Feature Importance Constraints

XGBoost provides built-in mechanisms to **hard-limit** how features interact and contribute:

#### Interaction Constraints

Restrict which features can appear together in the same branch:

```python
# Isolate 'income' so it can only interact with itself
interaction_constraints = [[0], [1, 2, 3, 4, 5]]  # Group indices
```

This means:
- `income` (index 0) forms splits **independently** — other features cannot appear below it in the same branch
- All other features can interact freely with each other

#### Monotone Constraints

Force the model's prediction to be monotonically increasing or decreasing with respect to a feature:

$$\text{monotone\_constraints} = (1, 0, 0, ...) \implies \frac{\partial \hat{y}}{\partial x_1} \geq 0 \text{ always}$$

This limits the complexity of splits on the constrained feature, reducing its importance.

#### `colsample_bytree` as Implicit Dropout

Setting `colsample_bytree = 0.6` means each tree randomly samples only 60% of features. On average, the dominant feature is **excluded from 40% of trees**, forcing the ensemble to learn alternative pathways.

In [0]:
# 5.6 Interaction Constraints: Isolate the dominant feature

# Feature indices: 0=age, 1=income, 2=education_years, 3=credit_score, 4=num_products, 5=tenure_months

# Constraint: income (idx 1) can ONLY interact with itself
# All other features can interact freely
model_constrained = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42,
    eval_metric='logloss',
    interaction_constraints=[[1], [0, 2, 3, 4, 5]]  # income isolated
)
model_constrained.fit(X_train, y_train)
auc_constrained = roc_auc_score(y_test, model_constrained.predict_proba(X_test)[:, 1])

# Monotone constraint on income (must be monotonically positive)
model_monotone = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42,
    eval_metric='logloss',
    monotone_constraints={'income': 1}  # income must increase prediction
)
model_monotone.fit(X_train, y_train)
auc_monotone = roc_auc_score(y_test, model_monotone.predict_proba(X_test)[:, 1])

# Compare importance distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_to_compare = [
    ('Baseline (no constraints)', model, 'coral'),
    ('Interaction Constrained', model_constrained, 'steelblue'),
    ('Monotone Constrained', model_monotone, 'seagreen')
]

for idx, (name, m, color) in enumerate(models_to_compare):
    imp = m.get_booster().get_score(importance_type='gain')
    total = sum(imp.values())
    imp_norm = {k: v/total for k, v in imp.items()}
    sorted_imp = dict(sorted(imp_norm.items(), key=lambda x: x[1]))
    
    colors = ['coral' if k == 'income' else color for k in sorted_imp.keys()]
    axes[idx].barh(list(sorted_imp.keys()), list(sorted_imp.values()), color=colors)
    axes[idx].set_title(name, fontsize=11)
    axes[idx].set_xlim(0, 1)

plt.suptitle('Feature Constraints: Limiting Income\'s Dominance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nPerformance comparison:")
print(f"  Baseline:                AUC = {auc_a:.4f}")
print(f"  Interaction constrained: AUC = {auc_constrained:.4f}")
print(f"  Monotone constrained:    AUC = {auc_monotone:.4f}")
print(f"\n\u2705 Constraints reduce income's dominance while preserving most performance.")
print(f"   The model is forced to extract more value from other features.")

### 5.7 Dimensionality Reduction on Correlated Feature Groups

When the dominant feature is **correlated** with several other features, the dominance may stem from a shared latent factor. PCA distributes the variance more evenly across orthogonal components.

#### Mathematical Formulation

Given a matrix of correlated features $$\mathbf{X} \in \mathbb{R}^{n \times p}$$, PCA finds eigenvectors $$\mathbf{v}_1, ..., \mathbf{v}_k$$ of the covariance matrix:

$$\mathbf{C} = \frac{1}{n-1} \mathbf{X}^T \mathbf{X}$$

The principal components are:

$$\text{PC}_j = \mathbf{X} \cdot \mathbf{v}_j$$

Each PC captures a decreasing proportion of variance:

$$\text{Explained variance ratio}_j = \frac{\lambda_j}{\sum_{i=1}^{p} \lambda_i}$$

#### Why this helps:

- If `income` correlates with `education_years` and `credit_score`, PCA creates components where the shared variance is **distributed** across PC1, PC2, PC3
- No single component can dominate unless the original features are almost identical
- The tree now splits on orthogonal signals rather than redundant ones

In [0]:
# 5.7 PCA: Distributing variance when features are correlated
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Create a scenario with correlated features (income-related cluster)
np.random.seed(42)
df_pca = df.copy()

# Add features that are correlated with income
df_pca['disposable_income'] = df_pca['income'] * 0.7 + np.random.normal(0, 5000, len(df_pca))
df_pca['savings'] = df_pca['income'] * 0.3 + np.random.normal(0, 3000, len(df_pca))
df_pca['spending_score'] = df_pca['income'] * 0.001 + np.random.normal(50, 10, len(df_pca))

# Check correlations
income_features = ['income', 'disposable_income', 'savings', 'spending_score']
print("Correlation matrix of income-related features:")
print(df_pca[income_features].corr().round(3).to_string())

# Apply PCA to the correlated group
scaler = StandardScaler()
income_scaled = scaler.fit_transform(df_pca[income_features])

pca = PCA(n_components=4)
pca_components = pca.fit_transform(income_scaled)

print(f"\nExplained variance by each PC:")
for i, var in enumerate(pca.explained_variance_ratio_):
    bar = '█' * int(var * 50)
    print(f"  PC{i+1}: {var:.3f} {bar}")

# Replace income cluster with PCA components
for i in range(4):
    df_pca[f'income_PC{i+1}'] = pca_components[:, i]

# Train model: Original vs PCA-transformed
features_pca = ['age', 'income_PC1', 'income_PC2', 'income_PC3', 'income_PC4',
                'education_years', 'credit_score', 'num_products', 'tenure_months']
features_corr = ['age', 'income', 'disposable_income', 'savings', 'spending_score',
                 'education_years', 'credit_score', 'num_products', 'tenure_months']

X_corr = df_pca[features_corr]
X_pca_features = df_pca[features_pca]
y_pca = df_pca['churn']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_corr, y_pca, test_size=0.2, random_state=42)
X_tr_p, X_te_p, _, _ = train_test_split(X_pca_features, y_pca, test_size=0.2, random_state=42)

# Model with correlated features
m_corr = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
m_corr.fit(X_tr_c, y_tr_c)
auc_corr = roc_auc_score(y_te_c, m_corr.predict_proba(X_te_c)[:, 1])

# Model with PCA components
m_pca = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
m_pca.fit(X_tr_p, y_tr_c)
auc_pca = roc_auc_score(y_te_c, m_pca.predict_proba(X_te_p)[:, 1])

# Compare importances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp1 = m_corr.get_booster().get_score(importance_type='gain')
t1 = sum(imp1.values())
imp1_n = {k: v/t1 for k, v in imp1.items()}
sorted1 = dict(sorted(imp1_n.items(), key=lambda x: x[1]))
axes[0].barh(list(sorted1.keys()), list(sorted1.values()), color='coral')
axes[0].set_title('Correlated Features\n(income cluster dominates)', fontsize=11)

imp2 = m_pca.get_booster().get_score(importance_type='gain')
t2 = sum(imp2.values())
imp2_n = {k: v/t2 for k, v in imp2.items()}
sorted2 = dict(sorted(imp2_n.items(), key=lambda x: x[1]))
axes[1].barh(list(sorted2.keys()), list(sorted2.values()), color='seagreen')
axes[1].set_title('PCA Components\n(variance distributed)', fontsize=11)

plt.suptitle('PCA: Distributing Correlated Feature Variance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nAUC (correlated features): {auc_corr:.4f}")
print(f"AUC (PCA components):      {auc_pca:.4f}")
print(f"\n\u2705 PCA distributes the income-cluster's dominance across orthogonal components.")
print(f"   Max importance dropped from {max(imp1_n.values()):.1%} to {max(imp2_n.values()):.1%}.")

### 5.8 SHAP-Based Analysis — Better Measurement, Not Correction

SHAP (SHapley Additive exPlanations) provides **theoretically grounded** feature importance based on cooperative game theory.

#### The Shapley Value

For feature $$j$$, the Shapley value is:

$$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[ f(S \cup \{j\}) - f(S) \right]$$

This computes the **marginal contribution** of feature $$j$$ averaged over all possible coalitions of other features.

#### Why SHAP is more reliable than Gain:

| Property | Gain-based | SHAP |
| --- | --- | --- |
| Additivity | ✘ (doesn't sum to prediction) | ✔ ($$\sum \phi_j = f(x) - E[f(x)]$$) |
| Cardinality bias | ✔ (inflates high-cardinality) | ✘ (unbiased) |
| Local explanations | ✘ (global only) | ✔ (per-instance) |
| Consistency | ✘ (can decrease when feature becomes more important) | ✔ (guaranteed consistent) |

#### When Gain ≠ SHAP:

If a feature shows **high gain but low SHAP**, it's likely:
- High cardinality (many splits, each with small effect)
- Correlated with another feature (redundant splits)
- Overfitting to noise in the training data

In [0]:
# 5.8 SHAP: Unbiased Feature Importance
import shap

# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Compare Gain vs SHAP importance
gain_imp = model.get_booster().get_score(importance_type='gain')
total_gain = sum(gain_imp.values())
gain_norm = {k: v/total_gain for k, v in gain_imp.items()}

# SHAP importance = mean absolute SHAP value per feature
shap_importance = np.abs(shap_values).mean(axis=0)
shap_total = shap_importance.sum()
shap_norm = {features[i]: shap_importance[i]/shap_total for i in range(len(features))}

# Side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gain
sorted_gain = dict(sorted(gain_norm.items(), key=lambda x: x[1]))
axes[0].barh(list(sorted_gain.keys()), list(sorted_gain.values()), color='coral')
axes[0].set_title('Gain-Based Importance\n(biased by cardinality & splits)', fontsize=11)
axes[0].set_xlabel('Normalized Importance')

# SHAP
sorted_shap = dict(sorted(shap_norm.items(), key=lambda x: x[1]))
axes[1].barh(list(sorted_shap.keys()), list(sorted_shap.values()), color='purple')
axes[1].set_title('SHAP Importance\n(theoretically unbiased)', fontsize=11)
axes[1].set_xlabel('Mean |SHAP value| (normalized)')

plt.suptitle('Gain vs SHAP: Which Features Truly Matter?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# SHAP summary plot (beeswarm)
print("\nSHAP Beeswarm Plot (shows direction + magnitude of each feature's impact):")
shap.summary_plot(shap_values, X_test, feature_names=features, show=True)

print("\nComparison Table:")
print(f"{'Feature':<20} {'Gain %':>8} {'SHAP %':>8} {'Discrepancy':>12}")
print("-"*50)
for feat in features:
    g = gain_norm.get(feat, 0)
    s = shap_norm.get(feat, 0)
    disc = g - s
    flag = " ⚠\ufe0f" if abs(disc) > 0.1 else ""
    print(f"{feat:<20} {g:>7.1%} {s:>7.1%} {disc:>+11.1%}{flag}")

print(f"\n\u2605 Large discrepancy (⚠\ufe0f) = feature importance inflated by cardinality/splits, not true signal.")

---
## 6. The High Cardinality Problem — Why It Causes False Dominance

### Definition

**High cardinality** refers to a feature having a very large number of unique values:
- `user_id`: 1,000,000 unique values
- `zip_code`: 40,000 unique values
- `product_sku`: 50,000 unique values
- `ip_address`: effectively infinite

### The Statistical Mechanism

Tree-based models split nodes by evaluating **all possible thresholds** for each feature. The more unique values a feature has, the more candidate split points it offers:

$$\text{Candidate splits for feature } k = |\text{unique}(x_k)| - 1$$

#### The Multiple Testing Problem:

With more candidate splits, the probability of finding a "good" split **by pure chance** increases:

$$P(\text{at least one split with gain} > \tau) = 1 - (1 - p)^{n_{\text{splits}}}$$

where $$p$$ is the probability that a random split achieves gain > $$\tau$$, and $$n_{\text{splits}}$$ is the number of candidate splits.

For a feature with 1,000 unique values: $$n_{\text{splits}} = 999$$  
For a binary feature: $$n_{\text{splits}} = 1$$

Even if both features carry equal **true** signal, the high-cardinality feature has 999× more chances to find a spuriously good split.

### Why Gain-Based Importance Is Biased

Gain importance sums the impurity reduction across **all splits** using a feature:

$$\text{Importance}_{\text{gain}}(k) = \sum_{t \in \text{Trees}} \sum_{\text{node } n \text{ splits on } k} \text{Gain}(n)$$

A high-cardinality feature:
1. Is selected for splits more often (more chances to win the split competition)
2. Each split may have very few samples (overfitting to noise)
3. Accumulates gain across many small, memorization-like splits

### The Memorization Trap

Consider `customer_id` with 500,000 unique values:
- Each ID maps to exactly 1 (or very few) samples
- A split on `customer_id = 12345` perfectly classifies that customer
- This is **memorization**, not generalization
- The tree essentially creates a lookup table

$$\text{Train accuracy} \approx 100\% \quad \text{but} \quad \text{Test accuracy} \approx \text{random}$$

In [0]:
# 6. Demonstrating the High Cardinality Problem

np.random.seed(42)
n = 5000

# Create features with different cardinalities but SAME true predictive power
df_card = pd.DataFrame({
    # Low cardinality: 5 categories (encoded as integers)
    'category_5': np.random.randint(0, 5, n),
    # Medium cardinality: 50 categories
    'category_50': np.random.randint(0, 50, n),
    # High cardinality: 500 categories
    'category_500': np.random.randint(0, 500, n),
    # Very high cardinality: 2000 categories  
    'category_2000': np.random.randint(0, 2000, n),
    # Near-unique: almost 1 per sample
    'near_unique_id': np.random.randint(0, 4000, n),
})

# Target is RANDOM - no feature has any true predictive power!
# Any apparent importance is purely due to overfitting/chance
df_card['target'] = np.random.randint(0, 2, n)

print("\u26a0\ufe0f IMPORTANT: The target is COMPLETELY RANDOM!")
print("   No feature has any true predictive power.")
print("   Any importance that appears is a STATISTICAL ARTIFACT.\n")

# Train XGBoost
features_card = ['category_5', 'category_50', 'category_500', 'category_2000', 'near_unique_id']
X_card = df_card[features_card]
y_card = df_card['target']

X_tr_card, X_te_card, y_tr_card, y_te_card = train_test_split(X_card, y_card, test_size=0.2, random_state=42)

model_card = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, 
                                random_state=42, eval_metric='logloss')
model_card.fit(X_tr_card, y_tr_card)

# Evaluate
auc_card_train = roc_auc_score(y_tr_card, model_card.predict_proba(X_tr_card)[:, 1])
auc_card_test = roc_auc_score(y_te_card, model_card.predict_proba(X_te_card)[:, 1])

# Get importance
imp_card = model_card.get_booster().get_score(importance_type='gain')
total_card = sum(imp_card.values())
imp_card_norm = {k: v/total_card for k, v in imp_card.items()}

# Get cover-based importance for comparison
imp_cover = model_card.get_booster().get_score(importance_type='cover')
total_cover = sum(imp_cover.values())
imp_cover_norm = {k: v/total_cover for k, v in imp_cover.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gain-based (biased)
cardinalities = {'category_5': 5, 'category_50': 50, 'category_500': 500, 
                 'category_2000': 2000, 'near_unique_id': 4000}
sorted_gain_card = dict(sorted(imp_card_norm.items(), key=lambda x: cardinalities.get(x[0], 0)))
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(sorted_gain_card)))
axes[0].barh(list(sorted_gain_card.keys()), list(sorted_gain_card.values()), color=colors)
axes[0].set_title('GAIN-Based Importance\n(BIASED: higher cardinality = higher importance)', fontsize=10)
axes[0].set_xlabel('Normalized Importance')

# Cover-based (less biased)
sorted_cover_card = dict(sorted(imp_cover_norm.items(), key=lambda x: cardinalities.get(x[0], 0)))
axes[1].barh(list(sorted_cover_card.keys()), list(sorted_cover_card.values()), color='steelblue')
axes[1].set_title('COVER-Based Importance\n(Less biased: samples affected per split)', fontsize=10)
axes[1].set_xlabel('Normalized Importance')

plt.suptitle('HIGH CARDINALITY BIAS: Random Target, Yet Gain Shows False Importance!', 
             fontsize=12, fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()

print(f"\n{'='*65}")
print(f"KEY RESULT: Target is RANDOM, yet model shows spurious importance:")
print(f"{'='*65}")
print(f"\n{'Feature':<20} {'Cardinality':>12} {'Gain Importance':>16} {'Cover Importance':>16}")
print(f"{'-'*66}")
for feat in sorted(features_card, key=lambda x: cardinalities[x]):
    g = imp_card_norm.get(feat, 0)
    c = imp_cover_norm.get(feat, 0)
    print(f"{feat:<20} {cardinalities[feat]:>12,} {g:>15.1%} {c:>15.1%}")

print(f"\nTrain AUC: {auc_card_train:.4f} (overfitting!)")
print(f"Test AUC:  {auc_card_test:.4f} (random = 0.50)")
print(f"\n\u2757 The model memorizes high-cardinality features on train, but gets")
print(f"   ~0.50 AUC on test (random). This PROVES the importance is an artifact.")

### 6.1 Detecting High Cardinality Bias

#### Red Flags:

1. **Gain >> SHAP**: The feature shows high gain-importance but low SHAP importance
2. **High gain, low cover**: Many splits but each affects very few samples
3. **Train-test gap**: Model overfits (high train AUC, low test AUC)
4. **Cardinality >> other features**: Order of magnitude more unique values

#### Detection Formula:

$$\text{Cardinality Bias Score} = \frac{\text{Gain Rank} - \text{SHAP Rank}}{p - 1}$$

where $$p$$ is the number of features. A score close to 1.0 means the feature's gain importance is massively inflated relative to its true contribution.

---

### 6.2 Solutions for High Cardinality

| Solution | How it works | When to use |
| --- | --- | --- |
| **Frequency encoding** | Replace category with its count in training data | Ordinal relationship with frequency |
| **Target encoding + smoothing** | Replace with smoothed target mean (Section 5.4) | Target relationship exists |
| **Hashing trick** | Map $$k$$ categories to $$h$$ buckets ($$h \ll k$$) | Very large cardinality, tolerate collisions |
| **Rare category grouping** | Bin categories below threshold into "OTHER" | Long-tail distribution |
| **Embedding (neural)** | Learn dense representation | When using neural networks |

In [0]:
# 6.2 Solutions for High Cardinality

np.random.seed(42)
n = 5000

# Create a realistic high-cardinality feature WITH actual signal
# merchant_id: 500 unique merchants, some with higher churn rates
n_merchants = 500
merchant_churn_rates = np.random.beta(2, 5, n_merchants)  # Variable churn rates

merchant_ids = np.random.randint(0, n_merchants, n)
other_feature = np.random.normal(0, 1, n)

# Target based on merchant's intrinsic churn rate + noise
prob_churn = merchant_churn_rates[merchant_ids] * 0.7 + 0.15 * (other_feature > 0)
target_hc = (np.random.rand(n) < prob_churn).astype(int)

df_hc = pd.DataFrame({
    'merchant_id': merchant_ids,
    'other_feature': other_feature,
    'target': target_hc
})

print("="*70)
print("SOLUTIONS FOR HIGH CARDINALITY (500 unique merchants)")
print("="*70)

# Solution 1: Frequency Encoding
freq_map = df_hc['merchant_id'].value_counts().to_dict()
df_hc['merchant_freq'] = df_hc['merchant_id'].map(freq_map)

# Solution 2: Target Encoding with smoothing
global_mean_hc = df_hc['target'].mean()
merchant_stats = df_hc.groupby('merchant_id')['target'].agg(['mean', 'count'])
smoothng = 20
df_hc['merchant_te'] = df_hc['merchant_id'].map(
    (merchant_stats['count'] * merchant_stats['mean'] + smoothng * global_mean_hc) / 
    (merchant_stats['count'] + smoothng)
)

# Solution 3: Hashing Trick (reduce 500 categories to 32 buckets)
def hash_feature(values, n_buckets=32):
    return np.array([hash(str(v)) % n_buckets for v in values])

df_hc['merchant_hashed'] = hash_feature(df_hc['merchant_id'], n_buckets=32)

# Solution 4: Rare category grouping (group merchants with < 5 samples)
merchant_counts = df_hc['merchant_id'].value_counts()
rare_merchants = merchant_counts[merchant_counts < 5].index
df_hc['merchant_grouped'] = df_hc['merchant_id'].apply(
    lambda x: -1 if x in rare_merchants else x  # -1 = "OTHER"
)
print(f"\nOriginal cardinality: {df_hc['merchant_id'].nunique()}")
print(f"After grouping rare:  {df_hc['merchant_grouped'].nunique()} (collapsed {len(rare_merchants)} rare merchants into 'OTHER')")
print(f"After hashing (32):   32 buckets")

# Compare all solutions
from sklearn.model_selection import cross_val_score

encoding_configs = {
    'Raw ID (problematic)': ['merchant_id', 'other_feature'],
    'Frequency Encoding': ['merchant_freq', 'other_feature'],
    'Target Encoding (m=20)': ['merchant_te', 'other_feature'],
    'Hash Buckets (32)': ['merchant_hashed', 'other_feature'],
}

print(f"\n{'Encoding Method':<30} {'Mean CV AUC':>12} {'Cardinality':>12}")
print("-"*56)

for name, feats in encoding_configs.items():
    X_enc = df_hc[feats].values
    y_enc = df_hc['target'].values
    m_enc = xgb.XGBClassifier(n_estimators=50, max_depth=4, learning_rate=0.1, 
                               random_state=42, eval_metric='logloss')
    cv_scores = cross_val_score(m_enc, X_enc, y_enc, cv=5, scoring='roc_auc')
    cardinality = df_hc[feats[0]].nunique()
    print(f"{name:<30} {cv_scores.mean():>12.4f} {cardinality:>12,}")

print(f"\n\u2705 Target encoding gives the best AUC with cardinality of 1 (continuous).")
print(f"   It captures the merchant-level signal without the memorization risk.")
print(f"   Frequency encoding is a safe baseline that requires no target information.")

---
## 7. Summary & Decision Framework

### The Complete Decision Tree for Handling Dominant Features

```
Feature dominates importance plot
│
├── Step 1: Is it DATA LEAKAGE?
│   ├── YES → REMOVE the feature immediately
│   └── NO → Continue...
│
├── Step 2: Is it a HIGH-CARDINALITY artifact?
│   ├── Check: Gain >> SHAP? Train AUC >> Test AUC?
│   ├── YES → Apply: Target encoding, Hashing, or Rare grouping
│   └── NO → Continue...
│
├── Step 3: Is the dominance LEGITIMATE but FRAGILE?
│   ├── Does performance drop significantly without it?
│   ├── YES → Feature carries real signal. Choose strategy:
│   │   ├── Want robustness → Feature Decomposition (5.1)
│   │   ├── Want diversity → Regularization + colsample (5.2)
│   │   ├── Want interpretability → Binning (5.3) or Constraints (5.6)
│   │   └── Want to understand residual value → Two-stage modeling (5.5)
│   └── NO → Feature can be removed or heavily penalized
│
└── Step 4: VALIDATE with SHAP (5.8)
    └── Always cross-check Gain vs SHAP to confirm your diagnosis
```

### Quick Reference: When to Use Each Technique

| Technique | Primary Use Case | Preserves Accuracy? | Complexity |
| --- | --- | --- | --- |
| Feature Decomposition | Aggregate feature with sub-components | ✔ High | Medium |
| Regularization (L1/L2/colsample) | Quick reduction without feature engineering | ✔ Medium-High | Low |
| Feature Binning | Continuous feature with too-fine resolution | ~ Moderate | Low |
| Target Encoding | High-cardinality categorical | ✔ High | Medium |
| Residual Modeling | Understanding "what else matters" | ✔ High | High |
| Interaction Constraints | Hard isolation in production | ~ Moderate | Low |
| PCA | Correlated feature group | ~ Moderate | Medium |
| SHAP Analysis | Diagnosis (not correction) | N/A | Low |

### Key Takeaways

1. **Don't panic** — dominance isn't always bad. Some features genuinely carry most of the signal.
2. **Always diagnose first** — leakage, cardinality bias, and genuine signal require different solutions.
3. **Feature crossing is rarely needed for trees** — save it for linear models.
4. **SHAP is your truth detector** — Gain lies, SHAP doesn't (for cardinality bias).
5. **Robustness > raw accuracy** — in production, a model that relies on one feature is fragile.
6. **Combine techniques** — regularization + decomposition + SHAP validation is a powerful stack.

---
*Notebook created as a comprehensive reference for handling feature dominance in gradient boosting models.*

In [0]:
# Final Summary: Compare all techniques applied to our original dataset

print("\n" + "="*75)
print(" COMPREHENSIVE COMPARISON: All Techniques on the Churn Dataset")
print("="*75)

summary_results = [
    ("Baseline (income dominates)", auc_a, max(imp_orig.values())),
    ("Without income (removed)", auc_b, None),
    ("Feature Decomposition (5.1)", auc_decomposed, max_dec),
    ("Regularization: colsample=0.5 (5.2)", None, None),  # Will recalculate
    ("Interaction Constraints (5.6)", auc_constrained, None),
    ("Monotone Constraints (5.6)", auc_monotone, None),
    ("Two-Stage Residual (5.5)", auc_combined, None),
]

# Recalculate colsample model
m_col = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                          random_state=42, eval_metric='logloss', colsample_bytree=0.5)
m_col.fit(X_train, y_train)
auc_col = roc_auc_score(y_test, m_col.predict_proba(X_test)[:, 1])
imp_col = m_col.get_booster().get_score(importance_type='gain')
t_col = sum(imp_col.values())
max_col = max(v/t_col for v in imp_col.values())

print(f"\n{'Technique':<42} {'AUC-ROC':>8} {'Max Feature %':>14} {'Dominance Δ':>12}")
print(f"{'-'*78}")

baseline_max = max(imp_orig.values())

results_final = [
    ("Baseline (income dominates)", auc_a, baseline_max),
    ("Feature removed (income dropped)", auc_b, None),
    ("Feature Decomposition (5.1)", auc_decomposed, max_dec),
    ("Regularization: colsample=0.5 (5.2)", auc_col, max_col),
    ("Interaction Constraints (5.6)", auc_constrained, None),
    ("Monotone Constraints (5.6)", auc_monotone, None),
    ("Two-Stage Residual (5.5)", auc_combined, None),
]

for name, auc, max_imp in results_final:
    auc_str = f"{auc:.4f}" if auc else "N/A"
    imp_str = f"{max_imp:.1%}" if max_imp else "N/A"
    delta = f"{(max_imp - baseline_max):.1%}" if max_imp else "N/A"
    print(f"{name:<42} {auc_str:>8} {imp_str:>14} {delta:>12}")

print(f"\n{'='*75}")
print(f"\n\u2705 All techniques successfully reduce dominance while preserving predictive power.")
print(f"   The best approach depends on your specific use case:")
print(f"   - Production robustness → Feature Decomposition + Regularization")
print(f"   - Interpretability → Binning + Constraints")
print(f"   - Understanding signal → Residual Modeling + SHAP")
print(f"   - High cardinality fix → Target Encoding + Hashing")